In [1]:
import os
import cv2
import torch
from PIL import Image
from tqdm import tqdm
import sys
import types
import torchvision.transforms.functional as F

# Fix for some library versions
sys.modules['torchvision.transforms.functional_tensor'] = F

# Model imports
from realesrgan import RealESRGANer
from gfpgan import GFPGANer
from facenet_pytorch import MTCNN
import cv2
import numpy as np


/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Input (contains image folders) and output directories
INPUT_DIR = '../testing_preprocess/'
OUTPUT_DIR = '../testing_output'

# The upscale factor for GFPGAN (usually 2).
UPSCALE_FACTOR = 2

# The margin to add around the detected face, as a percentage of face width.
# For example, 0.5 means a 50% margin.
DYNAMIC_MARGIN_PERCENT = 0.5

# Number of images to process in a single batch for face detection.
# Adjust based on your GPU VRAM. (16, 32, or 64 are common values).
BATCH_SIZE = 32

In [3]:
def setup_models(scale=2):
    """Initializes and prepares all required models."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    print("Setting up face detector (MTCNN)...")
    face_detector = MTCNN(keep_all=True, device=device, select_largest=False)

    print("Setting up face restorer (GFPGAN)...")
    gfpgan_model_path = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
    face_upsampler = GFPGANer(
        model_path=gfpgan_model_path,
        upscale=scale,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )

    print("All models are ready.")
    return face_detector, face_upsampler

In [4]:
def align_face(image, landmarks):
    """Rotates a cropped face image to make the eyes horizontal."""
    left_eye = landmarks[0]
    right_eye = landmarks[1]
    delta_x = right_eye[0] - left_eye[0]
    delta_y = right_eye[1] - left_eye[1]
    angle = np.degrees(np.arctan2(delta_y, delta_x))

    eye_center = (
        int((left_eye[0] + right_eye[0]) // 2),
        int((left_eye[1] + right_eye[1]) // 2)
    )

    h, w = image.shape[:2]
    rotation_matrix = cv2.getRotationMatrix2D(eye_center, angle, scale=1)
    aligned_image = cv2.warpAffine(image, rotation_matrix, (w, h))
    return aligned_image

In [5]:
def process_directory(input_dir, output_dir, detector, upscaler, margin_percent, batch_size):
    """
    Processes images using an 'Upscale First' workflow with batched face detection.
    1. Sequentially enhances each full image using GFPGAN.
    2. Performs face detection on a batch of these enhanced images.
    3. Crops, aligns, and saves the final faces.
    """
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    total_faces_saved = 0

    for root, _, files in os.walk(input_dir):
        image_paths = [os.path.join(root, f) for f in files if f.lower().endswith(image_extensions)]
        if not image_paths: continue

        print(f"\n[INFO] Found {len(image_paths)} images in '{os.path.basename(root)}'.")

        for i in tqdm(range(0, len(image_paths), batch_size), desc=f"Processing Batches in '{os.path.basename(root)}'"):
            batch_paths = image_paths[i:i + batch_size]

            try:
                # Step 1: Sequentially enhance each full image in the batch.
                # This part cannot be batched due to library limitations.
                restored_images_bgr = []
                for path in batch_paths:
                    img = cv2.imread(path, cv2.IMREAD_COLOR)
                    if img is not None:
                        _, _, restored_full = upscaler.enhance(img, has_aligned=False, only_center_face=False, paste_back=True)
                        restored_images_bgr.append(restored_full)

                if not restored_images_bgr: continue

                # Step 2: Perform face detection on the batch of enhanced images.
                pil_images = [Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)) for img in restored_images_bgr]
                all_boxes, _, all_landmarks = detector.detect(pil_images, landmarks=True)

                # Step 3: Process and save the results for each image in the batch.
                for j, (boxes, landmarks) in enumerate(zip(all_boxes, all_landmarks)):
                    if boxes is None: continue

                    restored_full_img = restored_images_bgr[j]
                    filename = os.path.basename(batch_paths[j])
                    output_subdir = os.path.join(output_dir, os.path.relpath(root, input_dir))
                    os.makedirs(output_subdir, exist_ok=True)

                    for k, (box, landmark) in enumerate(zip(boxes, landmarks)):
                        x1, y1, x2, y2 = [int(c) for c in box]
                        h, w, _ = restored_full_img.shape
                        face_width = x2 - x1
                        margin = int(face_width * margin_percent)

                        x1_m, y1_m = max(0, x1 - margin), max(0, y1 - margin)
                        x2_m, y2_m = min(w, x2 + margin), min(h, y2 + margin)

                        cropped_face = restored_full_img[y1_m:y2_m, x1_m:x2_m]
                        if cropped_face.size == 0: continue

                        adjusted_landmarks = landmark - np.array([x1_m, y1_m])
                        aligned_face = align_face(cropped_face, adjusted_landmarks)

                        base, _ = os.path.splitext(filename)
                        output_path = os.path.join(output_subdir, f"{base}_face_{k}.png")
                        cv2.imwrite(output_path, aligned_face)
                        total_faces_saved += 1

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"\n[ERROR] An exception occurred on batch starting with {os.path.basename(batch_paths[0])}: {e}")

    print(f"\nProcessing complete! Total faces saved: {total_faces_saved}")

In [6]:
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("PyTorch CUDA cache has been cleared successfully.")
    print(f"Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Memory Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
else:
    print("CUDA is not available. No GPU cache to clear.")

PyTorch CUDA cache has been cleared successfully.
Memory Allocated: 0.00 MB
Memory Reserved: 0.00 MB


In [7]:
if os.path.isdir(INPUT_DIR):
    face_detector, upsampler = setup_models(scale=UPSCALE_FACTOR)

    print("\nStarting optimized crop and upscale process...")
    face_detector, face_upsampler = setup_models(scale=UPSCALE_FACTOR)
    process_directory(
            INPUT_DIR,
            OUTPUT_DIR,
            face_detector,
            face_upsampler,
            DYNAMIC_MARGIN_PERCENT,
            BATCH_SIZE
        )
    print("\nProcessing complete!")
else:
    print(f"Error: Input directory not found at '{INPUT_DIR}'")

Using device: cuda
Setting up face detector (MTCNN)...
Setting up face restorer (GFPGAN)...


/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


All models are ready.

Starting optimized crop and upscale process...
Using device: cuda
Setting up face detector (MTCNN)...
Setting up face restorer (GFPGAN)...
All models are ready.

[INFO] Found 2583 images in '20241209091213'.


Processing Batches in '20241209091213': 100%|██████████| 81/81 [17:29<00:00, 12.95s/it]



[INFO] Found 2638 images in '20241209091346'.


...


[INFO] Found 2700 images in '20241212124240'.


Processing Batches in '20241212124240': 100%|██████████| 85/85 [18:40<00:00, 13.18s/it]



[INFO] Found 2598 images in '20241212125311'.


Processing Batches in '20241212125311': 100%|██████████| 82/82 [18:08<00:00, 13.28s/it]


Processing complete! Total faces saved: 74388

Processing complete!
